# Create Packard Fellowships for Science and Engineering Awards (FELLOWSHIP PATTERN)

Ingest of the David and Lucile Packard Foundation's official Fellows Directory at `www.packard.org/approach/fellowships-for-science-engineering/fellows-directory/`. The source directory advertises **730 fellows** across **1988-2025**, paginated as static server-rendered HTML. Each fellow card links to a detail page with cohort year, disciplines, current institution, fellowship institution when published, profile text, achievements, image, and canonical URL.

**Awarding body / funder:** David and Lucile Packard Foundation — `F4320306079`.

**Pattern: FELLOWSHIP.** Each row is one named Packard Fellow. `lead_investigator` is person-level: the fellow themselves, with `given_name` and `family_name` parsed from the official detail-page heading. `affiliation.name` maps to the source's `Fellowship Institution` field when present, not the current institution, because the fellowship institution is the award-time affiliation field the source explicitly publishes.

**Amount/currency waiver:** the current program overview says fellows receive **USD 875,000 over five years**, but the official directory does not publish per-fellow or historical award amounts. The source also spans 1988-2025, and applying today's amount backward would be an inference. This notebook therefore leaves `amount` and `currency` NULL by source authority and keeps the current program amount only in raw audit fields (`program_current_amount_usd`, `program_amount_note`, `program_amount_source_url`).

**Source authority:** Packard's own website; method #5 on the runbook ladder (static HTML scrape). No Wikipedia/Wikidata backfill.

**Prerequisites:** run `scripts/local/packard_fellows_to_s3.py` to crawl all directory pages, write `packard_fellows.parquet`, and upload to S3. Contractor handoff note: local validation can run with `--skip-upload`; admin must upload/run in Databricks because contractor has no S3/Databricks credentials.

**S3 location:** `s3a://openalex-ingest/awards/packard/packard_fellows.parquet`


## Step 1: Create staging table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.packard_fellows_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/packard/packard_fellows.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS total_raw_rows FROM openalex.awards.packard_fellows_raw;

## Step 1.5: Inspect raw data first

Expected source columns include `funder_award_id`, `slug`, `full_name`, `given_name`, `family_name`, `fellowship_year`, `disciplines`, `disciplines_text`, `current_institution`, `fellowship_institution`, `profile_text`, `achievements`, `display_name`, `description`, `start_date`, `end_date`, `amount`, `currency`, `program_current_amount_usd`, `program_amount_note`, `program_amount_source_url`, `profile_url`, `image_url`, `source_html_title`, `declined`, and `downloaded_at`.

Money scan note: the raw file intentionally carries NULL `amount`/`currency` for OpenAlex mapping. `program_current_amount_usd=875000` is an audit note from the current overview page, not a per-fellow historical amount column.


In [ ]:
%sql
DESCRIBE openalex.awards.packard_fellows_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.packard_fellows_raw LIMIT 10;

In [ ]:
%sql
SELECT
    MIN(TRY_CAST(program_current_amount_usd AS DOUBLE)) AS min_program_current_amount_usd,
    MAX(TRY_CAST(program_current_amount_usd AS DOUBLE)) AS max_program_current_amount_usd,
    COUNT(program_current_amount_usd) AS rows_with_program_note,
    COUNT(amount) AS rows_with_mapped_amount,
    COUNT(currency) AS rows_with_mapped_currency,
    COUNT(*) AS total_rows
FROM openalex.awards.packard_fellows_raw;

In [ ]:
%sql
SELECT
    MIN(TRY_CAST(fellowship_year AS INT)) AS min_year,
    MAX(TRY_CAST(fellowship_year AS INT)) AS max_year,
    COUNT(DISTINCT fellowship_year) AS distinct_years,
    COUNT(*) AS total_rows,
    COUNT(full_name) AS has_name,
    COUNT(disciplines_text) AS has_disciplines,
    COUNT(fellowship_institution) AS has_fellowship_institution,
    COUNT(profile_text) AS has_profile_text
FROM openalex.awards.packard_fellows_raw;

## Step 1.6: Funder existence fail-fast

In [ ]:
%sql
-- Must return exactly 1 row. If 0 rows, STOP: do not run the transform.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306079;

## Step 2: Transform to OpenAlex award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.packard_fellows_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306079
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'fellowship' AS funding_type,
    'Packard Fellowships for Science and Engineering' AS funder_scheme,
    'packard_fellows_directory' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date, 'yyyy-MM-dd') AS end_date,
    TRY_CAST(n.fellowship_year AS INT) AS start_year,
    TRY_CAST(n.fellowship_year AS INT) + 4 AS end_year,
    CASE
        WHEN n.full_name IS NULL OR n.full_name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.fellowship_institution AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.profile_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.packard_fellows_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.full_name IS NOT NULL;

In [ ]:
%sql
SELECT * FROM openalex.awards.packard_fellows_awards LIMIT 10;

## Step 3: Insert into `openalex_awards_raw` at priority 94

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'packard_fellows_directory' AND priority = 94;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    94 AS priority
FROM openalex.awards.packard_fellows_awards;

## Step 6: Verification

Expected local extraction: 730 rows from the official directory, 1988-2025. `amount`/`currency` are intentionally NULL with the waiver documented above; do not fail the ingest on amount coverage for this fellowship-directory source.


In [ ]:
%sql
SELECT COUNT(*) AS total_packard_awards FROM openalex.awards.packard_fellows_awards;

In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_lead_investigator,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) AS pct_lead_investigator
FROM openalex.awards.packard_fellows_awards;

In [ ]:
%sql
-- Amount/currency waiver: expected 0 populated because source does not publish historical per-fellow amounts.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    collect_set(currency) AS currencies
FROM openalex.awards.packard_fellows_awards;

In [ ]:
%sql
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.packard_fellows_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.packard_fellows_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.packard_fellows_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
SELECT id,
       SUBSTRING(display_name, 1, 100) AS title,
       lead_investigator.given_name AS given_name,
       lead_investigator.family_name AS family_name,
       lead_investigator.affiliation.name AS fellowship_institution,
       start_year,
       landing_page_url
FROM openalex.awards.packard_fellows_awards
ORDER BY start_year DESC, family_name
LIMIT 20;